# 01 - Setup

Install dependencies, download GSM8K, verify GPU, and smoke test Qwen3-4B via vLLM.

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/11-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"Repo: {REPO_DIR}")

Already up to date.
Workspace: /workspace/10-4-2026
Repo: /workspace/10-4-2026/legibility


In [2]:
!pip install -q vllm transformers datasets tqdm matplotlib seaborn

## Download GSM8K

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(f"GSM8K test split: {len(ds['test'])} examples")
print(f"First example:")
print(f"  Question: {ds['test'][0]['question'][:200]}...")
print(f"  Answer: {ds['test'][0]['answer'][:200]}...")

## Verify GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Smoke Test: Qwen3-4B via vLLM

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_NAME, dtype="bfloat16", max_model_len=4096)
sampling = SamplingParams(temperature=0.0, max_tokens=256)

test_prompt = "What is 2 + 3? Answer with just the number."
outputs = llm.generate([test_prompt], sampling)
print(f"Prompt: {test_prompt}")
print(f"Response: {outputs[0].outputs[0].text}")
print("\nSmoke test passed!")

In [ ]:
# Clean up to free GPU memory
del llm
import gc; gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")